# Notebook de documentacion, tratamiento datos y entrenamiento


## Equipo
- Alumno 1 : Maximiliano Romano 
- Alumno 2 : Facundo Molina

In [ ]:
## TO-DO utiliza esta notebook para documentar, entrenar y probar el modelo.

**LIBRERÍAS**

In [ ]:
import cv2
import os
from pathlib import Path
from sklearn.datasets import fetch_lfw_people
import numpy as np
import insightface
from PIL import Image
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple


**DATASET**

In [5]:
# DESCARGA DE DATASET LFW, RECORTE DE 20 PRINCIPALES Y GUARDADO EN CARPETA
lfw_people = fetch_lfw_people(min_faces_per_person=40, resize=1.0)

unique, counts = np.unique(lfw_people.target, return_counts=True)

top_20 = unique[np.argsort(counts)[-20:]]

mask = np.isin(lfw_people.target, top_20)
X = lfw_people.images[mask]
y = lfw_people.target[mask]


new_labels = {old: new for new, old in enumerate(top_20)}
y = np.array([new_labels[label] for label in y])

target_names = lfw_people.target_names[top_20]

base_path = Path(r"C:\Users\fjm25\Desktop\Facundo\TUIA\CV\tp1\tuia-face-recognition-app\data\processed")


for name in target_names:
    person_path = base_path / name.replace(" ", "_")
    person_path.mkdir(parents=True, exist_ok=True)

for i, (img, label_idx) in enumerate(zip(X, y)):

    name = target_names[label_idx].replace(" ", "_")

    img_to_save = (img * 255).astype(np.uint8)

    img_to_save = cv2.cvtColor(img_to_save, cv2.COLOR_GRAY2BGR)

    file_name = f"{name}_{i:04d}.jpg"
    file_path = base_path / name / file_name

    cv2.imwrite(str(file_path), img_to_save)

**DETECCIÓN Y ALINEACIÓN**

In [15]:
# ALINEACIÓN, DETECCIÓN, RECONOCIMIENTO, EMBEDDING CON INSIGHTFACE. MÁS RÁPIDA Y APLICA TODO EN UN PASO

dataset = Path('src/data/processed')
pil_faces = {}
output_dir = Path("output/post-processed")
output_dir.mkdir(parents=True, exist_ok=True)

embeddings_db = {}

for folder in dataset.iterdir():
    if folder.is_dir():
        persona_name = folder.name  
        
      
        for img_path in folder.glob('*.jpg'): 
                          
                pil_img = Image.open(img_path).convert('RGB')
                
                
                k = f"{persona_name}_{img_path.name}"

for name, img in k.items():

    face_data = service.detector(img) 
    
    if face_data is not None:

        embedding = service.get_embedding(face_data)
        
        embeddings_db[name] = embedding

"""for k, pil_img in dataset.items():
    # 1. Convertir PIL a formato compatible con OpenCV (BGR)
    img_array = np.array(pil_img.convert("RGB"))
    img_bgr = cv2.cvtColor(img_array, cv2.COLOR_RGB2BGR)"""

    # 2. Detección
boxes, _ = service.detect_faces(img_bgr)
    
if not boxes:
        print(f"No face detected for {k}")
        # Guardamos la imagen original redimensionada como fallback
        fallback = pil_img.resize((service.face_size, service.face_size))
        pil_faces[k] = fallback
        fallback.save(output_dir / f"{k}_noface.jpg")
else:
        for i, box in enumerate(boxes):
            # 3. Alineación
            aligned_obj = service.align_face(img_bgr, box)
            if aligned_obj is None: continue
            
            # Convertir de BGR (OpenCV) a RGB (PIL) para el diccionario y guardar
            face_rgb = cv2.cvtColor(aligned_obj.image, cv2.COLOR_BGR2RGB)
            face_pil = Image.fromarray(face_rgb)
            
            face_id = f"{k}_{i}"
            pil_faces[face_id] = face_pil
            face_pil.save(output_dir / f"{face_id}.jpg")
            
           
            embedding = service.extract_embedding_from_face(aligned_obj)
            service.store.append(...)

    

AttributeError: 'str' object has no attribute 'items'

**ARQUITECTURA(CON INCEPTIONRESNETV1)**

**EXTRACCIÓN DE EMBEDDINGS**

**VERIFICACIÓN**

**FINE-TUNING**

**GUARDADO**

In [ ]:
# from src.lib.schemas import AlignedFace, EmbeddingRecord, InsertRequest